# Lab 03B — استخدام الأدوات والأمان

**نظرة عامة:** تنفيذ أدوات المستشفى بشكل آمن من خلال التحقق من الصلاحيات، تأكيد الإجراءات عالية الخطورة، تحديد عدد خطوات الـTool Loop، وتصعيد الحالات الحرجة للبشر.

راجع ملف `README.md` داخل هذا المجلد للحصول على شرح مختصر ومنظم للأهداف، طريقة العمل، النتائج، وأهم النقاط التعليمية.


In [1]:
ASSISTANT_NAME = "Farah Hospital AI"
CURRENT_USER = "Farah"

PATIENT_RECORDS = {
    "FH1001": {
        "name": "Farah",
        "city": "Khobar",
        "department": "orthopedics",
        "priority": "URGENT"
    },

    "FH1002": {
        "name": "Sara",
        "city": "Riyadh",
        "department": "cardiology",
        "priority": "CRITICAL"
    },

    "FH1003": {
        "name": "Ahmed",
        "city": "Dammam",
        "department": "dermatology",
        "priority": "NORMAL"
    }
}

print("🏥", ASSISTANT_NAME)
print("Current User:", CURRENT_USER)
print("Registered Patients:", len(PATIENT_RECORDS))

🏥 Farah Hospital AI
Current User: Farah
Registered Patients: 3


In [2]:
def check_patient_record(patient_id):
    record = PATIENT_RECORDS.get(patient_id)

    if record is None:
        return "Patient record not found"

    return (
        "Patient: " + record["name"] +
        " | Department: " + record["department"] +
        " | Priority: " + record["priority"]
    )


def check_doctor_availability(department, city):
    availability = {
        ("cardiology", "Riyadh"): "Dr. Omar - 4:00 PM",
        ("orthopedics", "Khobar"): "Dr. Hani - 2:30 PM",
        ("dermatology", "Dammam"): "Dr. Sara - 6:00 PM",
        ("general_medicine", "Qatif"): "Dr. Ali - 5:00 PM"
    }

    return availability.get(
        (department, city),
        "No available appointment found"
    )


def book_hospital_appointment(name, department, city, date):
    return (
        "✅ Appointment confirmed for " + name +
        " | Department: " + department +
        " | City: " + city +
        " | Date: " + date
    )


def emergency_alert(patient_name, symptom):
    return (
        "🚨 EMERGENCY ALERT CREATED for " +
        patient_name +
        " | Reason: " + symptom
    )


def escalate_to_medical_staff(reason):
    return (
        "👩‍⚕️ Case transferred to medical staff | Reason: "
        + reason
    )

In [3]:
TOOLS = {

    "check_patient_record": {
        "function": check_patient_record,
        "risk": "read_only"
    },

    "check_doctor_availability": {
        "function": check_doctor_availability,
        "risk": "read_only"
    },

    "book_hospital_appointment": {
        "function": book_hospital_appointment,
        "risk": "side_effecting"
    },

    "emergency_alert": {
        "function": emergency_alert,
        "risk": "high_risk"
    },

    "escalate_to_medical_staff": {
        "function": escalate_to_medical_staff,
        "risk": "terminal"
    }
}


print("===== FARAH HOSPITAL TOOL REGISTRY =====")

for name, tool in TOOLS.items():
    print(name, "→", tool["risk"])

===== FARAH HOSPITAL TOOL REGISTRY =====
check_patient_record → read_only
check_doctor_availability → read_only
book_hospital_appointment → side_effecting
emergency_alert → high_risk
escalate_to_medical_staff → terminal


In [4]:
def run_tool(name, args):

    if name not in TOOLS:
        return "❌ ERROR: Unknown tool"

    tool = TOOLS[name]

    # Protect actions that change hospital state
    if tool["risk"] == "side_effecting":

        if args.get("name") != CURRENT_USER:
            return "🔒 ERROR: User is not authorized"

    # High-risk medical actions need explicit confirmation
    if tool["risk"] == "high_risk":

        confirmed = args.pop("confirmed", False)

        if confirmed is not True:
            return "⚠️ ERROR: Emergency action requires confirmation"

    try:
        return tool["function"](**args)

    except TypeError:
        return "❌ ERROR: Wrong arguments"

In [5]:
print("===== AUTHORIZATION TEST =====")

# Test 1: Read patient record
print(
    run_tool(
        "check_patient_record",
        {"patient_id": "FH1001"}
    )
)

print()

# Test 2: Authorized booking for Farah
farah_booking = {
    "name": "Farah",
    "department": "orthopedics",
    "city": "Khobar",
    "date": "2026-10-10"
}

print(
    run_tool(
        "book_hospital_appointment",
        farah_booking
    )
)

print()

# Test 3: Unauthorized booking for another user
unauthorized_booking = {
    "name": "Fahad",
    "department": "cardiology",
    "city": "Riyadh",
    "date": "2026-10-10"
}

print(
    run_tool(
        "book_hospital_appointment",
        unauthorized_booking
    )
)

===== AUTHORIZATION TEST =====
Patient: Farah | Department: orthopedics | Priority: URGENT

✅ Appointment confirmed for Farah | Department: orthopedics | City: Khobar | Date: 2026-10-10

🔒 ERROR: User is not authorized


In [6]:
print("===== EMERGENCY SAFETY TEST =====")

emergency_request = {
    "patient_name": "Sara",
    "symptom": "severe chest pain",
    "confirmed": False
}

print(
    run_tool(
        "emergency_alert",
        emergency_request
    )
)

===== EMERGENCY SAFETY TEST =====
⚠️ ERROR: Emergency action requires confirmation


In [7]:
confirmed_emergency = {
    "patient_name": "Sara",
    "symptom": "severe chest pain and difficulty breathing",
    "confirmed": True
}

print(
    run_tool(
        "emergency_alert",
        confirmed_emergency
    )
)

🚨 EMERGENCY ALERT CREATED for Sara | Reason: severe chest pain and difficulty breathing


In [8]:
MAX_ITERATIONS = 6


def run_with_tools(steps):

    print("🤖 Starting Farah Hospital AI Tool Loop")
    print("-" * 60)

    for iteration in range(1, MAX_ITERATIONS + 1):

        if iteration > len(steps):
            break

        step = steps[iteration - 1]

        if "text" in step:
            return step["text"]

        tool_name = step.get("tool")
        args = step.get("args", {}).copy()

        result = run_tool(tool_name, args)

        print(
            "Iteration",
            iteration,
            "|",
            tool_name,
            "→",
            result
        )

        # Stop immediately after terminal escalation
        if tool_name == "escalate_to_medical_staff":
            return result

    return (
        "⚠️ Maximum tool iterations reached. "
        "Transferring case to medical staff."
    )

In [9]:
appointment_steps = [

    {
        "tool": "check_patient_record",
        "args": {
            "patient_id": "FH1001"
        }
    },

    {
        "tool": "check_doctor_availability",
        "args": {
            "department": "orthopedics",
            "city": "Khobar"
        }
    },

    {
        "tool": "book_hospital_appointment",
        "args": {
            "name": "Farah",
            "department": "orthopedics",
            "city": "Khobar",
            "date": "2026-10-10"
        }
    },

    {
        "text":
        "✅ Your orthopedic appointment has been arranged successfully."
    }
]


final_response = run_with_tools(appointment_steps)

print()
print("AI FINAL RESPONSE:")
print(final_response)

🤖 Starting Farah Hospital AI Tool Loop
------------------------------------------------------------
Iteration 1 | check_patient_record → Patient: Farah | Department: orthopedics | Priority: URGENT
Iteration 2 | check_doctor_availability → Dr. Hani - 2:30 PM
Iteration 3 | book_hospital_appointment → ✅ Appointment confirmed for Farah | Department: orthopedics | City: Khobar | Date: 2026-10-10

AI FINAL RESPONSE:
✅ Your orthopedic appointment has been arranged successfully.


In [10]:
critical_steps = [

    {
        "tool": "check_patient_record",
        "args": {
            "patient_id": "FH1002"
        }
    },

    {
        "tool": "emergency_alert",
        "args": {
            "patient_name": "Sara",
            "symptom":
            "severe chest pain and difficulty breathing",
            "confirmed": True
        }
    },

    {
        "tool": "escalate_to_medical_staff",
        "args": {
            "reason":
            "Critical symptoms require immediate human assessment"
        }
    }
]


result = run_with_tools(critical_steps)

print()
print("FINAL STATUS:")
print(result)

🤖 Starting Farah Hospital AI Tool Loop
------------------------------------------------------------
Iteration 1 | check_patient_record → Patient: Sara | Department: cardiology | Priority: CRITICAL
Iteration 2 | emergency_alert → 🚨 EMERGENCY ALERT CREATED for Sara | Reason: severe chest pain and difficulty breathing
Iteration 3 | escalate_to_medical_staff → 👩‍⚕️ Case transferred to medical staff | Reason: Critical symptoms require immediate human assessment

FINAL STATUS:
👩‍⚕️ Case transferred to medical staff | Reason: Critical symptoms require immediate human assessment


In [12]:
print("===== NEGATIVE / SAFETY TESTS =====")


tests = [

    {
        "name": "Unknown Tool Attack",
        "tool": "delete_all_patient_records",
        "args": {}
    },

    {
        "name": "Unauthorized Booking",
        "tool": "book_hospital_appointment",
        "args": {
            "name": "Fahad",
            "department": "cardiology",
            "city": "Riyadh",
            "date": "2026-10-10"
        }
    },

    {
        "name": "Wrong Arguments",
        "tool": "check_patient_record",
        "args": {
            "number": "FH1001"
        }
    },

    {
        "name": "Unconfirmed Emergency",
        "tool": "emergency_alert",
        "args": {
            "patient_name": "Sara",
            "symptom": "chest pain",
            "confirmed": False
        }
    }
]


for test in tests:

    print()
    print("TEST:", test["name"])

    result = run_tool(test["tool"], test["args"])
    print("Result:", result)

===== NEGATIVE / SAFETY TESTS =====

TEST: Unknown Tool Attack
Result: ❌ ERROR: Unknown tool

TEST: Unauthorized Booking
Result: 🔒 ERROR: User is not authorized

TEST: Wrong Arguments
Result: ❌ ERROR: Wrong arguments

TEST: Unconfirmed Emergency
Result: ⚠️ ERROR: Emergency action requires confirmation
